# Homework: Constraint Satisfaction Problems (CSPs)
## SCIS 432: Artificial Intelligence

In this assignment, you will implement a **Sudoku solver** as a **Constraint Satisfaction Problem (CSP)** using:

- **Backtracking search**
- **Arc Consistency (AC-3)**

You will be given starter code and will fill in key parts of the solver.

---
### Submission
Save as a pdf and submit with all code cells run.

### Rules
- You may use Python standard library and `numpy` (already imported).
- Do **not** use external Sudoku/CSP libraries.


## Learning Objectives

By the end of this assignment, you should be able to:

- Represent a problem as a CSP (variables, domains, constraints)
- Implement a **backtracking** CSP solver
- Enforce **arc consistency** using **AC-3**
- (Optional) Add CSP heuristics (MRV, degree, LCV)


## Sudoku as a Constraint Satisfaction Problem (CSP)

In this assignment, we model a **4×4 Sudoku** as a constraint satisfaction problem.  
This is a simplified version of standard 9×9 Sudoku, chosen to make the CSP algorithms easier to implement, debug, and reason about.

A **4×4 Sudoku** has:
- A **4×4 grid** (16 total cells)
- Digits **1–4** (instead of 1–9)
- **2×2 subgrids** (blocks)

The rules are identical to standard Sudoku: each digit must appear **exactly once** in every row, column, and block.

### CSP Formulation

We define the Sudoku as a CSP with the following components:

- **Variables:**  
  One variable per cell, identified by its row and column `(r, c)`.

- **Domains:**  
  - If a cell is pre-filled in the puzzle, its domain is the singleton set `{given value}`.  
  - Otherwise, its domain is `{1, 2, 3, 4}`.

- **Constraints:**  
  All constraints are **pairwise inequality constraints**:
  - **Row constraint:** cells in the same row must have different values  
  - **Column constraint:** cells in the same column must have different values  
  - **Block constraint:** cells in the same 2×2 block must have different values  

Each cell is constrained to be different from all of its **peers** (cells sharing a row, column, or block).

In [ ]:
import numpy as np

# 0 means empty
PUZZLE_1 = np.array([
    [1, 0, 0, 4],
    [0, 0, 3, 0],
    [0, 3, 0, 0],
    [2, 0, 0, 1],
])

PUZZLE_2 = np.array([
    [0, 0, 2, 0],
    [0, 1, 0, 4],
    [1, 0, 0, 0],
    [0, 4, 0, 0],
])

def pretty_print(grid):
    """Print a 4x4 Sudoku grid."""
    g = np.array(grid)
    for r in range(4):
        if r in (2,):
            print("------+------")
        row = []
        for c in range(4):
            if c in (2,):
                row.append("|")
            v = int(g[r, c])
            row.append(str(v) if v != 0 else ".")
        print(" ".join(row))

print("Puzzle 1:")
pretty_print(PUZZLE_1)


## Helper Functions (Provided)

These are provided for you.


In [ ]:
# Helper utilities for 4x4 Sudoku

DIGITS = {1, 2, 3, 4}

def cells():
    return [(r, c) for r in range(4) for c in range(4)]

def block_index(r, c):
    """Return the (br, bc) index of the 2x2 block containing (r,c)."""
    return (r // 2, c // 2)

def block_cells(br, bc):
    """Cells in a 2x2 block."""
    return [(r, c) for r in range(br*2, br*2 + 2) for c in range(bc*2, bc*2 + 2)]

def peers_of(var):
    """Return all variables that share a row, column, or block with var, excluding var itself."""
    r, c = var
    peer_set = set()

    # same row
    for cc in range(4):
        if cc != c:
            peer_set.add((r, cc))

    # same column
    for rr in range(4):
        if rr != r:
            peer_set.add((rr, c))

    # same block
    br, bc = block_index(r, c)
    for v in block_cells(br, bc):
        if v != var:
            peer_set.add(v)

    return peer_set


## CSP Data Structures

We will represent a CSP with:

- `variables`: list of variables (cells)
- `domains`: dict mapping var -> set of possible values
- `neighbors`: dict mapping var -> set of peer variables
- `constraints`: implemented via a function `consistent(var, value, assignment)` that checks pairwise inequality

### Your Task
Complete `build_domains` and `consistent`.


In [ ]:
def build_domains(grid):
    """
    Given a 4x4 numpy array grid (0 = empty),
    return a dict: (r,c) -> set(domain values).

    - If grid[r,c] is nonzero, the domain should be {that value}.
    - Otherwise, the domain should start as {1,2,3,4}.

    NOTE: We will later prune domains using AC-3.
    """
    domains = {}
    for var in cells():
        r, c = var
        if grid[r, c] != 0:
            domains[var] = {int(grid[r, c])}
        else:
            # TODO: set initial domain for empty cell
            domains[var] = None
    return domains


def build_neighbors():
    """Return a dict var -> set(peers)."""
    return {v: peers_of(v) for v in cells()}


def consistent(var, value, assignment, neighbors):
    """
    Return True if assigning `value` to `var` does NOT violate any constraints
    with respect to the current partial assignment.

    Pairwise inequality constraints:
      For each neighbor n of var:
        if n is assigned, then assignment[n] != value

    assignment: dict var -> int
    """
    # TODO: implement constraint check
    raise NotImplementedError


## Arc Consistency (AC-3)

AC-3 enforces *arc consistency* by pruning domains.

For Sudoku with inequality constraints, an arc `(Xi, Xj)` is consistent if for every value in `domain[Xi]` there exists some value in `domain[Xj]` that does not conflict.

For inequality, a value `x` in `Xi`'s domain is supported by `Xj` if `Xj` has **some** value `y != x`.

### Your Task
Complete `revise` and `ac3`.


In [ ]:
from collections import deque

def revise(domains, Xi, Xj):
    """
    Make domain of Xi arc-consistent with Xj.
    Remove values x from domains[Xi] that have no supporting y in domains[Xj].

    For inequality constraints: x is unsupported if domains[Xj] == {x}.

    Returns:
        revised (bool): True iff we removed something from Xi's domain
    """
    revised = False

    # TODO: implement revise for inequality constraint
    # Hint: iterate over a *copy* of domains[Xi] while removing from domains[Xi].

    return revised


def ac3(domains, neighbors):
    """
    AC-3 algorithm.

    Args:
        domains: dict var -> set of possible values (mutated in-place)
        neighbors: dict var -> set of neighboring vars

    Returns:
        (bool): True if arc consistency enforced successfully,
                False if some domain becomes empty (inconsistency).
    """
    queue = deque()
    # Initialize queue with all arcs
    for Xi in neighbors:
        for Xj in neighbors[Xi]:
            queue.append((Xi, Xj))

    while queue:
        Xi, Xj = queue.popleft()

        # TODO: if revise(domains, Xi, Xj) is True, then:
        #   - if domains[Xi] becomes empty, return False
        #   - for each Xk in neighbors[Xi] except Xj, add (Xk, Xi) back to the queue

        pass

    return True


## Backtracking Search

Now implement **backtracking** to find a complete assignment.

You should implement:
- `select_unassigned_variable` (baseline or MRV)
- `order_domain_values` (baseline or LCV)
- `backtrack`

Recommended order:
1) get baseline backtracking working (choose first unassigned, iterate domain in sorted order)
2) add MRV
3) add LCV

### Your Task
Complete the functions below.


In [ ]:
def is_complete(assignment):
    return len(assignment) == 16


def select_unassigned_variable(assignment, domains):
    """
    Pick the next unassigned variable.

    Baseline: first unassigned variable in row-major order.
    Optional (recommended): MRV = variable with smallest remaining domain.
    """
    # TODO
    raise NotImplementedError


def order_domain_values(var, domains, assignment, neighbors):
    """
    Return a list of values to try for var.

    Baseline: sorted(domains[var]).
    Optional: LCV ordering (least-constraining value).
    """
    # TODO
    raise NotImplementedError


def forward_check(domains, var, value, neighbors):
    """
    OPTIONAL helper: return a list of (neighbor, removed_value) pairs
    that were removed from neighbor domains when var=value is assigned.

    For inequality constraints:
      if neighbor domain contains 'value', remove it.

    If any neighbor domain becomes empty, return None.

    NOTE: This is optional—your solver can work without forward checking.
    """
    removed = []
    for n in neighbors[var]:
        if value in domains[n]:
            domains[n].remove(value)
            removed.append((n, value))
            if len(domains[n]) == 0:
                return None
    return removed


def restore(domains, removed):
    """Undo removals returned by forward_check."""
    for (v, val) in removed:
        domains[v].add(val)


def backtrack(assignment, domains, neighbors):
    """
    Backtracking search.

    Returns:
        assignment dict var->value if solved, else None.

    Guidance:
    - If complete, return assignment
    - Pick an unassigned var
    - For each value in ordered domain:
        - If consistent, assign
        - (Optional) forward check / domain pruning
        - Recurse
        - Undo assignment and restore domains if needed
    - Return None if no value works
    """
    # TODO
    raise NotImplementedError


## Putting It Together

Complete the `solve_sudoku` function:
- Build domains
- Build neighbors
- Run AC-3 once at the start to prune domains
- Run backtracking to finish

Then test on the provided puzzles.


In [ ]:
def solve_sudoku(grid):
    neighbors = build_neighbors()
    domains = build_domains(grid)

    # Sanity check: domains should be sets
    for v, d in domains.items():
        if not isinstance(d, set):
            raise TypeError(f"Domain for {v} is not a set. Got: {type(d)}")

    # Enforce arc consistency before search
    ok = ac3(domains, neighbors)
    if not ok:
        return None

    assignment = {}
    # Assign all singleton domains immediately (optional but helpful)
    for v, d in domains.items():
        if len(d) == 1:
            assignment[v] = next(iter(d))

    sol = backtrack(assignment, domains, neighbors)
    if sol is None:
        return None

    # Convert assignment to grid
    out = np.zeros((4,4), dtype=int)
    for (r,c), val in sol.items():
        out[r,c] = val
    return out


# --- tests ---
for i, puzzle in enumerate([PUZZLE_1, PUZZLE_2], start=1):
    print(f"\nSolving Puzzle {i}:")
    pretty_print(puzzle)
    sol = solve_sudoku(puzzle.copy())
    if sol is None:
        print("No solution found.")
    else:
        print("Solution:")
        pretty_print(sol)


## Extension (Optional)

Pick **one** extension for extra credit:

1. **Heuristics:** Add MRV + LCV and report how many assignments your solver makes with and without heuristics.
2. **Forward checking:** Implement forward checking inside backtracking (use `forward_check` / `restore`).
3. **9×9 Sudoku:** Generalize to 9×9 (digits 1–9, 3×3 blocks). Provide at least one puzzle.

For extra credit, include a short write-up in the reflection section.


## Reflection Questions

Answer in a few sentences each:

1. In your own words, what does AC-3 do to the domains, and why can it speed up search?
2. Give an example (from Sudoku or another CSP) where AC-3 would detect failure *without* needing full search.
3. If you implemented MRV/LCV/forward checking, which helped most on these puzzles and why?


## Grading Rubric (100 points)

- (15) `build_domains` correct
- (15) `consistent` correct
- (25) `revise` + `ac3` correct (prunes domains; detects inconsistency)
- (35) `backtrack` solver works on provided puzzles
- (10) Code clarity: readable, commented where needed, reasonable variable names

Extra credit: up to +10
